In [17]:
import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.


In [18]:
path

'/kaggle/input/brain-tumor-mri-dataset'

In [19]:
! cp -r /kaggle/input/brain-tumor-mri-dataset /content/Brain_Tumor

In [20]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , models

In [21]:
train_path = r"/content/Brain_Tumor/Training"
test_path =  r"/content/Brain_Tumor/Testing"

## Split The Data

In [38]:
img_size = (224,224)
batch_size = 32
seed = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    image_size=img_size,
    batch_size=batch_size,
    seed=seed)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split= 0.2,
    subset= "validation",
    image_size= img_size,
    batch_size= batch_size,
    seed= seed
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size= img_size,
    batch_size = batch_size
)


Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.


In [39]:
print("Number of classes: ", len(train_ds.class_names))
print("Class Names:", train_ds.class_names)

Number of classes:  4
Class Names: ['glioma', 'meningioma', 'notumor', 'pituitary']


## Autotune

In [24]:
Autotune = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=Autotune)
validation_ds = validation_ds.cache().prefetch(buffer_size=Autotune)
test_ds = test_ds.cache().prefetch(buffer_size=Autotune)

## Build The Model

In [25]:
# Augmenation

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),
    layers.RandomRotation(0.1)
])

In [26]:
# Model Layers

model = models.Sequential([
    layers.InputLayer(shape=(224,224,3)),

    augmentation,

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(4, activation='softmax')
])

In [27]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,826,884 (37.49 MB)

 Trainable params: 9,826,884 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
early_Stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

history = model.fit(
    train_ds,
    epochs=50,
    validation_data=validation_ds,
    callbacks=[early_Stopping],
    verbose=1
    )


Epoch 1/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 25s 87ms/step - accuracy: 0.5754 - loss: 0.9786 - val_accuracy: 0.7696 - val_loss: 0.6284
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 70ms/step - accuracy: 0.7261 - loss: 0.6932 - val_accuracy: 0.7991 - val_loss: 0.5027
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 65ms/step - accuracy: 0.7741 - loss: 0.5715 - val_accuracy: 0.7625 - val_loss: 0.5833
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 66ms/step - accuracy: 0.7944 - loss: 0.5175 - val_accuracy: 0.8777 - val_loss: 0.3321
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.8161 - loss: 0.4677 - val_accuracy: 0.8804 - val_loss: 0.3298
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.8275 - loss: 0.4496 - val_accuracy: 0.8768 - val_loss: 0.3188
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.8522 - loss: 0.3802 - val_accuracy: 0.8634 - val_loss: 0.3828
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 68ms/step - accuracy: 0.8533 - loss: 0.3721 - val_a

## Evaluation

In [29]:
loss , acc = model.evaluate(test_ds)
print("Accuracy: ", acc)

50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.9169 - loss: 0.8498
Accuracy:  0.9168750047683716


In [41]:
import numpy as np

img_path = "/content/Brain_Tumor/Testing/meningioma/Te-aug-me_20.jpg"
img_size = (224,224)

image = tf.keras.utils.load_img(
    img_path,
    target_size=img_size
)

img_arr = tf.keras.utils.img_to_array(image)
img_arr = np.expand_dims(img_arr,axis=0)

In [42]:
prediction = model.predict(img_arr)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
[[1.6174922e-03 9.9500436e-01 3.3776355e-03 5.4019949e-07]]


In [43]:
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

prediction_index = np.argmax(prediction)
predicted_class = class_names[prediction_index]

confidence_acc = prediction[0][prediction_index] * 100

print(f"Predicted Class: {predicted_class}")
print(f"Confidence: {confidence_acc:.2f}%")

Predicted Class: meningioma
Confidence: 99.50%


In [45]:
model.save("Brain_Tumor_MRI.keras")